In [22]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# New imports for XGBoost, Naive Bayes, and SVM
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.feature_selection import chi2, mutual_info_classif, SelectKBest
from sklearn.feature_selection import f_classif
from skrebate import ReliefF

import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


Load Dataset

- 48 word frequency attributes (word_freq_*)
- 6 character frequency attributes (char_freq_*)
- 3 capital run length attributes
- 1 class label (spam: 1, non-spam: 0)

In [23]:
df = pd.read_csv('Spam.csv')

print(f"Dataset shape: {df.shape}")

print(df['spam'].value_counts())

df.head()

Dataset shape: (4601, 58)
spam
0    2788
1    1813
Name: count, dtype: int64


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_semicolon,char_freq_paren,char_freq_bracket,char_freq_exclaim,char_freq_dollar,char_freq_hash,capital_run_length_average,capital_run_length_longest,capital_run_length_total,spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


Separate features and target

In [24]:
X = df.drop('spam', axis=1)
y = df['spam']

feature_names = X.columns.tolist()
print(f"Number of features: {len(feature_names)}")
print(f"Number of samples: {len(X)}")

Number of features: 57
Number of samples: 4601


Train-Test Split 

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

Training set: (3680, 57)
Testing set: (921, 57)


Feature Selection 
1. **Chi-square**: Measures dependency between features and class
2. **Information Gain**: Entropy-based measure
3. **Gain Ratio**: Normalized information gain
4. **Symmetrical Uncertainty**: Normalized mutual information
5. **Relief**: Context-sensitive feature weighting
6. **OneR**: Simple rule-based ranking
7. **Correlation**: Pearson correlation with class

define functions

In [26]:
def calculate_entropy(y):
    """Calculate entropy of a variable"""
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
    return entropy

def information_gain_score(X, y):
    """Calculate information gain for each feature"""
    scores = mutual_info_classif(X, y, random_state=42)
    return scores

def gain_ratio_score(X, y):
    """Calculate gain ratio for each feature"""
    ig_scores = information_gain_score(X, y)
    gr_scores = []
    
    for i in range(X.shape[1]):
        feature_entropy = calculate_entropy(X.iloc[:, i])
        if feature_entropy > 0:
            gr = ig_scores[i] / feature_entropy
        else:
            gr = 0
        gr_scores.append(gr)
    
    return np.array(gr_scores)

def symmetrical_uncertainty_score(X, y):
    """Calculate symmetrical uncertainty for each feature"""
    ig_scores = information_gain_score(X, y)
    y_entropy = calculate_entropy(y)
    su_scores = []
    
    for i in range(X.shape[1]):
        feature_entropy = calculate_entropy(X.iloc[:, i])
        su = 2.0 * ig_scores[i] / (y_entropy + feature_entropy + 1e-10)
        su_scores.append(su)
    
    return np.array(su_scores)

def correlation_score(X, y):
    """Calculate correlation for each feature with target"""
    scores = []
    for col in X.columns:
        corr = abs(np.corrcoef(X[col], y)[0, 1])
        scores.append(corr)
    return np.array(scores)

def oner_score(X, y):
    """OneR feature selection - simple rule-based scoring"""
    scores = []
    for i in range(X.shape[1]):
        feature = X.iloc[:, i]
        # Discretize continuous features into bins
        if len(np.unique(feature)) > 10:
            feature_binned = pd.cut(feature, bins=10, labels=False, duplicates='drop')
        else:
            feature_binned = feature
        
        # Calculate accuracy of simple rule
        accuracy = 0
        for val in np.unique(feature_binned):
            mask = feature_binned == val
            if mask.sum() > 0:
                most_common_class = y[mask].mode()[0] if len(y[mask].mode()) > 0 else 0
                accuracy += (y[mask] == most_common_class).sum()
        
        scores.append(accuracy / len(y))
    
    return np.array(scores)


Score calculation

In [27]:
# 1. Chi-square
chi2_scores, _ = chi2(X_train, y_train)
print("✓ Chi-square scores calculated")

# 2. Information Gain
ig_scores = information_gain_score(X_train, y_train)
print("✓ Information Gain scores calculated")

# 3. Gain Ratio
gr_scores = gain_ratio_score(X_train, y_train)
print("✓ Gain Ratio scores calculated")

# 4. Symmetrical Uncertainty
su_scores = symmetrical_uncertainty_score(X_train, y_train)
print("✓ Symmetrical Uncertainty scores calculated")

# 5. Relief
relief = ReliefF(n_neighbors=10, n_features_to_select=len(feature_names))
relief.fit(X_train.values, y_train.values)
relief_scores = relief.feature_importances_
print("✓ Relief scores calculated")

# 6. OneR
oner_scores = oner_score(X_train, y_train)
print("✓ OneR scores calculated")

# 7. Correlation
corr_scores = correlation_score(X_train, y_train)
print("✓ Correlation scores calculated")

print("\nAll feature selection methods completed!")

✓ Chi-square scores calculated
✓ Information Gain scores calculated
✓ Gain Ratio scores calculated
✓ Symmetrical Uncertainty scores calculated
✓ Relief scores calculated
✓ OneR scores calculated
✓ Correlation scores calculated

All feature selection methods completed!


In [28]:
feature_scores = pd.DataFrame({
    'Feature': feature_names,
    'Chi-square': chi2_scores,
    'InfoGain': ig_scores,
    'GainRatio': gr_scores,
    'SU': su_scores,
    'Relief': relief_scores,
    'OneR': oner_scores,
    'Correlation': corr_scores
})

for method in ['Chi-square', 'InfoGain', 'GainRatio', 'SU', 'Relief', 'OneR', 'Correlation']:
    top_features = feature_scores.nlargest(10, method)['Feature'].tolist()
    print(f"{method:15s}: {', '.join(top_features[:5])}...")

Chi-square     : capital_run_length_total, capital_run_length_longest, capital_run_length_average, word_freq_george, word_freq_hp...
InfoGain       : char_freq_dollar, char_freq_exclaim, capital_run_length_longest, word_freq_your, capital_run_length_average...
GainRatio      : word_freq_table, word_freq_remove, word_freq_money, word_freq_3d, word_freq_000...
SU             : word_freq_remove, char_freq_dollar, word_freq_money, word_freq_000, word_freq_free...
Relief         : word_freq_george, word_freq_hp, word_freq_will, word_freq_addresses, word_freq_direct...
OneR           : word_freq_your, word_freq_you, word_freq_000, word_freq_all, word_freq_receive...
Correlation    : word_freq_your, word_freq_000, word_freq_remove, char_freq_dollar, word_freq_business...


Select Subsets - top 87%, 77%, and 70% of features from each

In [48]:
def get_top_features(scores, feature_names, percentage):
    n_features = int(len(feature_names) * percentage / 100)
    top_indices = np.argsort(scores)[-n_features:]
    return [feature_names[i] for i in top_indices]

feature_subsets = {}
percentages = [95,90,85,80,75,70]

for pct in percentages:
    feature_subsets[pct] = {
        'Chi': get_top_features(chi2_scores, feature_names, pct),
        'InfoGain': get_top_features(ig_scores, feature_names, pct),
        'GainRatio': get_top_features(gr_scores, feature_names, pct),
        'SU': get_top_features(su_scores, feature_names, pct),
        'Relief': get_top_features(relief_scores, feature_names, pct),
        'OneR': get_top_features(oner_scores, feature_names, pct),
        'Corr': get_top_features(corr_scores, feature_names, pct)
    }
    print(f"{pct}% features: {len(feature_subsets[pct]['Chi'])} features selected")


95% features: 54 features selected
90% features: 51 features selected
85% features: 48 features selected
80% features: 45 features selected
75% features: 42 features selected
70% features: 39 features selected


### Random Forest Classifier

In [49]:
def train_random_forest(X_train, y_train, X_test, y_test):
    """Train Random Forest as per paper specifications"""
    start_time = time.time()

    rf = RandomForestClassifier(
        n_estimators=10,
        max_features=4,
        max_depth=None,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    train_time = (time.time() - start_time) * 1000  
    
    train_pred = rf.predict(X_train)
    test_pred = rf.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred) * 100
    test_acc = accuracy_score(y_test, test_pred) * 100
    
    return {
        'model': rf,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time_ms': train_time
    }


### Decision Tree Classifier


In [50]:
def train_part(X_train, y_train, X_test, y_test):
    start_time = time.time()

    part = DecisionTreeClassifier(
        criterion='entropy',
        splitter='best',
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    )
    
    part.fit(X_train, y_train)
    train_time = (time.time() - start_time) * 1000 

    train_pred = part.predict(X_train)
    test_pred = part.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred) * 100
    test_acc = accuracy_score(y_test, test_pred) * 100
    
    return {
        'model': part,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time_ms': train_time
    }

### XGBoost Classifier

In [51]:
def train_xgboost(X_train, y_train, X_test, y_test):
    """Train XGBoost Classifier"""
    start_time = time.time()
    
    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    
    xgb.fit(X_train, y_train)
    train_time = (time.time() - start_time) * 1000
    
    train_pred = xgb.predict(X_train)
    test_pred = xgb.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred) * 100
    test_acc = accuracy_score(y_test, test_pred) * 100
    
    return {
        'model': xgb,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time_ms': train_time
    }

### Naive Bayes Classifier

In [52]:

def train_naive_bayes(X_train, y_train, X_test, y_test):
    """Train Naive Bayes Classifier"""
    start_time = time.time()
    
    nb = GaussianNB()
    
    nb.fit(X_train, y_train)
    train_time = (time.time() - start_time) * 1000
    
    train_pred = nb.predict(X_train)
    test_pred = nb.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred) * 100
    test_acc = accuracy_score(y_test, test_pred) * 100
    
    return {
        'model': nb,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time_ms': train_time
    }

### SVM Classifier

In [53]:
def train_svm(X_train, y_train, X_test, y_test):
    """Train Support Vector Machine Classifier"""
    start_time = time.time()
    
    svm = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        random_state=42
    )
    
    svm.fit(X_train, y_train)
    train_time = (time.time() - start_time) * 1000
    
    train_pred = svm.predict(X_train)
    test_pred = svm.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred) * 100
    test_acc = accuracy_score(y_test, test_pred) * 100
    
    return {
        'model': svm,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time_ms': train_time
    }

Baseline

In [58]:
print("Random Forest:")
rf_baseline = train_random_forest(X_train, y_train, X_test, y_test)
print(f"  Training Accuracy: {rf_baseline['train_acc']:.3f}%")
print(f"  Testing Accuracy:  {rf_baseline['test_acc']:.3f}%")
print(f"  Training Time:     {rf_baseline['time_ms']:.0f} ms")

print("DecisionTrees")
part_baseline = train_part(X_train, y_train, X_test, y_test)
print(f"  Training Accuracy: {part_baseline['train_acc']:.3f}%")
print(f"  Testing Accuracy:  {part_baseline['test_acc']:.3f}%")
print(f"  Training Time:     {part_baseline['time_ms']:.0f} ms")

print("\nXGBoost")
xgb_baseline = train_xgboost(X_train, y_train, X_test, y_test)
print(f"  Training Accuracy: {xgb_baseline['train_acc']:.3f}%")
print(f"  Testing Accuracy:  {xgb_baseline['test_acc']:.3f}%")
print(f"  Training Time:     {xgb_baseline['time_ms']:.0f} ms")

print("\nNaive Bayes")
nb_baseline = train_naive_bayes(X_train, y_train, X_test, y_test)
print(f"  Training Accuracy: {nb_baseline['train_acc']:.3f}%")
print(f"  Testing Accuracy:  {nb_baseline['test_acc']:.3f}%")
print(f"  Training Time:     {nb_baseline['time_ms']:.0f} ms")

print("\nSVM")
svm_baseline = train_svm(X_train, y_train, X_test, y_test)
print(f"  Training Accuracy: {svm_baseline['train_acc']:.3f}%")
print(f"  Testing Accuracy:  {svm_baseline['test_acc']:.3f}%")
print(f"  Training Time:     {svm_baseline['time_ms']:.0f} ms")


Random Forest:
  Training Accuracy: 99.620%
  Testing Accuracy:  93.268%
  Training Time:     20 ms
DecisionTrees
  Training Accuracy: 99.973%
  Testing Accuracy:  91.965%
  Training Time:     26 ms

XGBoost
  Training Accuracy: 98.342%
  Testing Accuracy:  94.680%
  Training Time:     149 ms

Naive Bayes
  Training Accuracy: 82.255%
  Testing Accuracy:  83.388%
  Training Time:     1 ms

SVM
  Training Accuracy: 71.603%
  Testing Accuracy:  70.575%
  Training Time:     137 ms


Feature Selection

In [ ]:
results = []

for pct in percentages: 
    print(f"\n{'='*80}")
    print(f"RESULTS WITH {pct}% FEATURES ({len(feature_subsets[pct]['Chi'])} features)")
    print(f"{'='*80}\n")
    
    for fs_method in ['Chi', 'InfoGain', 'GainRatio', 'SU', 'Relief', 'OneR', 'Corr']:
        selected_features = feature_subsets[pct][fs_method]
        
        # Subset the data
        X_train_subset = X_train[selected_features]
        X_test_subset = X_test[selected_features]
        
        # Train Random Forest
        rf_result = train_random_forest(X_train_subset, y_train, X_test_subset, y_test)
        
        # Train Decision Tree
        part_result = train_part(X_train_subset, y_train, X_test_subset, y_test)
        
        # Train XGBoost
        xgb_result = train_xgboost(X_train_subset, y_train, X_test_subset, y_test)
        
        # Train Naive Bayes
        nb_result = train_naive_bayes(X_train_subset, y_train, X_test_subset, y_test)
        
        # Train SVM
        svm_result = train_svm(X_train_subset, y_train, X_test_subset, y_test)
        
        # Store results
        results.append({
            'FS_Percentage': pct,
            'FS_Method': fs_method,
            'RF_Train_Acc': rf_result['train_acc'],
            'RF_Test_Acc': rf_result['test_acc'],
            'RF_Time_ms': rf_result['time_ms'],
            'DT_Train_Acc': part_result['train_acc'],
            'DT_Test_Acc': part_result['test_acc'],
            'DT_Time_ms': part_result['time_ms'],
            'XGB_Train_Acc': xgb_result['train_acc'],
            'XGB_Test_Acc': xgb_result['test_acc'],
            'XGB_Time_ms': xgb_result['time_ms'],
            'NB_Train_Acc': nb_result['train_acc'],
            'NB_Test_Acc': nb_result['test_acc'],
            'NB_Time_ms': nb_result['time_ms'],
            'SVM_Train_Acc': svm_result['train_acc'],
            'SVM_Test_Acc': svm_result['test_acc'],
            'SVM_Time_ms': svm_result['time_ms']
        })
        
        print(f"{fs_method:12s} | RF: T={rf_result['train_acc']:.1f}% V={rf_result['test_acc']:.1f}% | "
              f"DT: T={part_result['train_acc']:.1f}% V={part_result['test_acc']:.1f}% | "
              f"XGB: T={xgb_result['train_acc']:.1f}% V={xgb_result['test_acc']:.1f}% | "
              f"NB: T={nb_result['train_acc']:.1f}% V={nb_result['test_acc']:.1f}% | "
              f"SVM: T={svm_result['train_acc']:.1f}% V={svm_result['test_acc']:.1f}%")

print("\n" + "="*80)
print("All experiments completed!")
print("="*80)


RESULTS WITH 95% FEATURES (54 features)

Chi          | RF: T=99.5% V=93.7% | DT: T=100.0% V=91.9% | XGB: T=98.3% V=94.6% | NB: T=82.2% V=83.3% | SVM: T=71.6% V=70.6%
InfoGain     | RF: T=99.3% V=93.5% | DT: T=100.0% V=91.6% | XGB: T=98.4% V=94.5% | NB: T=82.0% V=83.1% | SVM: T=71.6% V=70.6%
GainRatio    | RF: T=99.5% V=94.0% | DT: T=100.0% V=92.0% | XGB: T=98.3% V=94.6% | NB: T=82.0% V=83.3% | SVM: T=71.6% V=70.6%
SU           | RF: T=99.6% V=94.4% | DT: T=100.0% V=91.9% | XGB: T=98.4% V=94.8% | NB: T=82.1% V=83.4% | SVM: T=71.6% V=70.6%
Relief       | RF: T=99.5% V=93.6% | DT: T=100.0% V=91.6% | XGB: T=98.4% V=94.7% | NB: T=81.9% V=83.1% | SVM: T=71.6% V=70.6%
OneR         | RF: T=99.5% V=94.0% | DT: T=100.0% V=91.9% | XGB: T=98.5% V=95.0% | NB: T=82.3% V=84.0% | SVM: T=71.6% V=70.6%
Corr         | RF: T=99.5% V=94.5% | DT: T=100.0% V=91.5% | XGB: T=98.3% V=94.7% | NB: T=81.5% V=83.1% | SVM: T=71.6% V=70.6%

RESULTS WITH 90% FEATURES (51 features)

Chi          | RF: T=99.6% V=93.5%

In [56]:

print("Baseline (100% features):")
print(f"  Random Forest: Train={rf_baseline['train_acc']:.3f}% Test={rf_baseline['test_acc']:.3f}% Time={rf_baseline['time_ms']:.0f}ms")
print(f"  Decision Tree: Train={part_baseline['train_acc']:.3f}% Test={part_baseline['test_acc']:.3f}% Time={part_baseline['time_ms']:.0f}ms")
print(f"  XGBoost:       Train={xgb_baseline['train_acc']:.3f}% Test={xgb_baseline['test_acc']:.3f}% Time={xgb_baseline['time_ms']:.0f}ms")
print(f"  Naive Bayes:   Train={nb_baseline['train_acc']:.3f}% Test={nb_baseline['test_acc']:.3f}% Time={nb_baseline['time_ms']:.0f}ms")
print(f"  SVM:           Train={svm_baseline['train_acc']:.3f}% Test={svm_baseline['test_acc']:.3f}% Time={svm_baseline['time_ms']:.0f}ms")


Baseline (100% features):
  Random Forest: Train=99.620% Test=93.268% Time=16ms
  Decision Tree: Train=99.973% Test=91.965% Time=34ms
  XGBoost:       Train=98.342% Test=94.680% Time=147ms
  Naive Bayes:   Train=82.255% Test=83.388% Time=2ms
  SVM:           Train=71.603% Test=70.575% Time=141ms


In [ ]:

results_df = pd.DataFrame(results)

algorithms_info = {
    'RF': ('Random Forest', 'RF_Test_Acc', 'RF_Train_Acc', 'RF_Time_ms', rf_baseline),
    'DT': ('Decision Tree', 'DT_Test_Acc', 'DT_Train_Acc', 'DT_Time_ms', part_baseline),
    'XGB': ('XGBoost', 'XGB_Test_Acc', 'XGB_Train_Acc', 'XGB_Time_ms', xgb_baseline),
    'NB': ('Naive Bayes', 'NB_Test_Acc', 'NB_Train_Acc', 'NB_Time_ms', nb_baseline),
    'SVM': ('SVM', 'SVM_Test_Acc', 'SVM_Train_Acc', 'SVM_Time_ms', svm_baseline)
}


overall_best_by_alg = {}
for alg_code, (alg_name, test_col, train_col, time_col, baseline) in algorithms_info.items():
    best_fs_idx = results_df[test_col].idxmax()
    best_fs = results_df.loc[best_fs_idx]
    
    if best_fs[test_col] > baseline['test_acc']:
        overall_best_by_alg[alg_code] = {
            'name': alg_name,
            'test': best_fs[test_col],
            'train': best_fs[train_col],
            'time': best_fs[time_col],
            'pct': best_fs['FS_Percentage'],
            'method': best_fs['FS_Method']
        }
    else:
        overall_best_by_alg[alg_code] = {
            'name': alg_name,
            'test': baseline['test_acc'],
            'train': baseline['train_acc'],
            'time': baseline['time_ms'],
            'pct': 100,
            'method': 'Baseline'
        }

all_best_list = sorted(overall_best_by_alg.values(), key=lambda x: x['test'], reverse=True)

print("\nKey Findings:")
for alg_code in ['RF', 'DT', 'XGB', 'NB', 'SVM']:
    res = overall_best_by_alg[alg_code]
    print(f"• Best {res['name']}: {res['test']:.2f}% ({res['pct']}% features via {res['method']})")




Key Findings:
• Best Random Forest: 94.90% (90% features via Corr)
• Best Decision Tree: 92.83% (80% features via GainRatio)
• Best XGBoost: 95.22% (90% features via Chi)
• Best Naive Bayes: 86.75% (80% features via OneR)
• Best SVM: 73.18% (80% features via GainRatio)
